### 1. colab 연동

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import random

import torchvision
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary
from tqdm import tqdm
import torch.optim.lr_scheduler as lr_scheduler

Mounted at /content/drive


### 2. CIFAR100 data를 train, validation으로 나누기

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dir = 'drive/MyDrive/cifar100/train'
test_dir = 'drive/MyDrive/cifar100/test'
batch_size = 64

transform = transforms.Compose([
    # transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'Number of training samples: {len(train_dataset)}')
print(f'Number of testing samples: {len(test_dataset)}')
print(f'Number of classes: {len(train_dataset.classes)}')
print(f'Class names: {train_dataset.classes}')
print(f'Example image shape: {train_dataset[0][0].shape}')

Number of training samples: 50001
Number of testing samples: 10000
Number of classes: 100
Class names: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'te

### 3. 모델 정의


In [ ]:
import torch.nn as nn

class depthwise_separable_conv(nn.Module):
    def __init__(self,in_channels,out_channels,stride, activation=nn.ReLU, norm_layer=nn.BatchNorm2d, layer_suffle=False):
        super(depthwise_separable_conv,self).__init__()
        if layer_suffle:
            self.dconv = nn.Sequential(
                nn.Conv2d(in_channels,in_channels,3,stride,1,groups=in_channels),
                activation(),
                norm_layer(in_channels)
            )
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels,out_channels,1,1),
                activation(),
                norm_layer(out_channels)
            )
        else:
            self.dconv = nn.Sequential(
                nn.Conv2d(in_channels,in_channels,3,stride,1,groups=in_channels),
                norm_layer(in_channels),
                activation()
            )
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels,out_channels,1,1),
                norm_layer(out_channels),
                activation()
            )

        

    def forward(self,x):
        out = self.dconv(x)
        out = self.conv(out)

        return out


class MobileNet(nn.Module):
    def __init__(self,a=1, activation=nn.ReLU, norm_layer=nn.BatchNorm2d, layer_suffle=False):
        super(MobileNet,self).__init__()
        set_activation = activation
        set_norm_layer = norm_layer
        set_layer_suffle = layer_suffle

        self.conv1 = nn.Sequential(
            nn.Conv2d(3,32*a,3,2,1),
            norm_layer(32*a),
            activation()
        )

        self.Mobile = nn.Sequential(
            depthwise_separable_conv(32*a,64,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(64,128,2, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(128,128,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(128,256,2, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(256,256,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(256,512,2, set_activation, set_norm_layer, set_layer_suffle),

            depthwise_separable_conv(512,512,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(512,512,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(512,512,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(512,512,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(512,512,1, set_activation, set_norm_layer, set_layer_suffle),

            depthwise_separable_conv(512,1024,1, set_activation, set_norm_layer, set_layer_suffle),
            depthwise_separable_conv(1024,1024,1, set_activation, set_norm_layer, set_layer_suffle),
            nn.AdaptiveAvgPool2d(1)

        )

        self.FC = nn.Sequential(
            nn.Linear(1024,100)
        )

    def forward(self,x):
        out = self.conv1(x)
        out = self.Mobile(out)
        out = out.view(out.size(0),-1)
        out = self.FC(out)

        return out

In [14]:
summary(MobileNet().to(device), (3,32,32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 16, 16]             896
       BatchNorm2d-2           [-1, 32, 16, 16]              64
              ReLU-3           [-1, 32, 16, 16]               0
            Conv2d-4           [-1, 32, 16, 16]             320
       BatchNorm2d-5           [-1, 32, 16, 16]              64
              ReLU-6           [-1, 32, 16, 16]               0
            Conv2d-7           [-1, 64, 16, 16]           2,112
       BatchNorm2d-8           [-1, 64, 16, 16]             128
              ReLU-9           [-1, 64, 16, 16]               0
depthwise_separable_conv-10           [-1, 64, 16, 16]               0
           Conv2d-11             [-1, 64, 8, 8]             640
      BatchNorm2d-12             [-1, 64, 8, 8]             128
             ReLU-13             [-1, 64, 8, 8]               0
           Conv2d-14            

### 4. Data Augmentation

In [6]:
def soft_cross_entropy(pred, soft_targets):
    logsoftmax = nn.LogSoftmax(dim=1)
    return torch.mean(torch.sum(- soft_targets * logsoftmax(pred), 1))

def mixup(x, y, alpha = 1.0, num_classes=100):
    if alpha > 0:
        lambda_ = np.random.beta(alpha, alpha)
    else:
        lambda_ = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size)

    mixed_input = lambda_ * x + (1 - lambda_) * x[index, :]

    y_onehot = F.one_hot(y, num_classes=num_classes).float()
    y_return = lambda_ * y_onehot + (1 - lambda_) * y_onehot[index, :]

    return mixed_input, y_return

### 5. train, test 함수 정의

In [7]:
def train(dataloader , model , loss_fn , optimizer , lr_scheduler):
    size = 0
    num_batches = len(dataloader)

    model.train()
    epoch_loss , epoch_correct = 0 , 0

    for i ,(data_ , target_) in enumerate(dataloader):

        data_mixup , target_mixup = mixup(data_ , target_)
        data_ , target_ = data_.to(device), target_.to(device)
        data_mixup, target_mixup = data_mixup.to(device), target_mixup.to(device)
        optimizer.zero_grad()

        output = model(data_mixup)

        loss = loss_fn(output, target_mixup)
        loss.backward()
        optimizer.step()

        pred = output.argmax(dim=1)
        correct = (pred == target_).sum().item()
        epoch_correct += correct
        epoch_loss += loss.item()
        size += len(data_)

    train_acc = epoch_correct/size
    lr_scheduler.step()

    return train_acc , epoch_loss / num_batches

In [8]:
def test(dataloader , model , loss_fn):
    size = 0
    num_baches = len(dataloader)
    epoch_loss , epoch_correct= 0 ,0
    with torch.no_grad(): # grad 연산 X
        model.eval() # evaluation dropout 연산시
        for i, (data_ , target_) in enumerate(dataloader):

            data_ , target_ = data_.to(device), target_.to(device)
            optimizer.zero_grad()

            output = model(data_)
            output = torch.sigmoid(output.squeeze())

            loss = loss_fn(output, target_)

            pred = output.argmax(dim=1)
            correct = (pred == target_).sum().item()
            epoch_correct += correct
            epoch_loss += loss.item()
            size += len(data_)

    test_acc = epoch_correct/size

    return test_acc  , epoch_loss / num_baches

### 6. 학습 및 테스트

In [ ]:
EPOCHS = 100
logs = {"ReLU_acc":[] , 
        "sigmoid_acc":[] , 
        "batch_norm_acc":[] , 
        "group_norm_acc":[],
        "CBA_acc":[],
        "CRB_acc":[],
        "CLRB_acc":[]
        }

gn_layer = lambda channels: nn.GroupNorm(num_groups=4, num_channels=channels)
models = {
    "ReLU": MobileNet(activation=nn.ReLU).to(device),
    "sigmoid": MobileNet(activation=nn.Sigmoid).to(device),
    "batch_norm": MobileNet(norm_layer=nn.BatchNorm2d).to(device),
    "group_norm": MobileNet(norm_layer=gn_layer).to(device),
    "CBA": MobileNet(activation=nn.ReLU, norm_layer=nn.BatchNorm2d).to(device),
    "CRB": MobileNet(activation=nn.ReLU, norm_layer=nn.BatchNorm2d).to(device),
    "CLRB": MobileNet(activation=nn.LeakyReLU, norm_layer=nn.BatchNorm2d).to(device)
}
models_name = list(models.keys())

criterion = nn.CrossEntropyLoss()
criterion_mixup = soft_cross_entropy

for iteration in range(logs.__len__()):
    current_model = models[models_name[iteration]]
    optimizer = optim.SGD(current_model.parameters(), 1e-2, momentum=0.9, nesterov=True, weight_decay=5e-4)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    print('='*50)
    print(f'current_model: {list(models.keys())[iteration]}')
    print('='*50)
    for epoch in tqdm(range(EPOCHS)):
        train_acc, train_loss = train(train_loader ,
                                        current_model ,
                                        criterion_mixup ,
                                        optimizer ,
                                        scheduler)
        test_acc, test_loss = test(test_loader, current_model, criterion)
        current_lr = optimizer.param_groups[0]['lr']

        print('\n'f'train_acc:{train_acc:.4f} test_acc:{test_acc:.4f}')
        
        logs[iteration].append(train_loss)
        logs[iteration].append(test_acc)


KeyError: 0

## 8. 출력 및 저장

In [ ]:
def final_test(dataloader , model):
    pred_array = []

    with torch.no_grad():
        model.eval()
        for i, (data_) in enumerate(dataloader):
            data_= data_.to(device)
            output = model(data_)

            pred_array.append(output.argmax(dim=1))

    return pred_array

In [ ]:
df = final_test(test_loader, MyModel)
for i in range(len(df)):
    df[i] = df[i].cpu().numpy()
df = np.concatenate(df)

df = pd.DataFrame({'id_idx': np.arange(len(df)), 'label': df})
df.to_csv('drive/MyDrive/3-2 AI 집중교육/kaggle challenge/result_csv/kaggle_array_MobileNetV2_Dropout.csv', index=False)